In [1]:
from askslm import metadata, taxonomy
from askslm.contracts import SubsetCriteria, TaxaSelector, SampleSelector

# Metadata
metadata_df = metadata.extract("/Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/inputs/metadata.csv")
metadata_df.to_csv("out/metadata.csv", index=False)
overview = metadata.overview(metadata_df)
overview.write("out/metadata_overview")   # saves 5 CSVs

# Taxonomy: load → resolve → build abundance matrix
taxonomy_df = taxonomy.load("/Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/inputs/taxonomy.csv")
resolved    = taxonomy.resolve(taxonomy_df)
# uses bundled merged.dmp
matrix      = taxonomy.build_abundance_matrix(resolved.df, rank="genus")
matrix.write("out/abundance.tsv", fmt="maaslin")

# Subset: Bacteria clade, WGS assay, control subjects only
CONTROL_SUBJECTS = ("CAN", "CAC", "CAM", "CAK", "CAA")

subset = taxonomy.get_profile_subset(
    matrix, metadata_df,
    SubsetCriteria(
        taxa=TaxaSelector(rank="superkingdom", names=("Bacteria",)),
        sample_filters=(
            SampleSelector(column="assay_type", values=("WGS",)),
            SampleSelector(column="host_subject_id_sam", values=CONTROL_SUBJECTS),
        ),
    ),
)

subset.write("out/amplicon_and_control_excluded_subset")

print("Subset profile shape:", subset.profile_df.shape)
print("Control subjects in subset:", sorted(subset.metadata_df["host_subject_id_sam"].unique()))
print("Assay types in subset:", subset.metadata_df["assay_type"].unique().tolist())


Subset profile shape: (80, 3577)
Control subjects in subset: ['CAA', 'CAC', 'CAK', 'CAM', 'CAN']
Assay types in subset: ['WGS']
